## 👩‍💻 Author & Context

**Author:** Ana Laura Della Rosa  

This project was developed as part of a technical assessment for **PM Accelerator**.

PM Accelerator is a global organization focused on building AI-driven products and helping professionals gain hands-on experience by working on real-world projects in collaborative environments.

# 🌍 Weather Travel Assistant (Backend)

This project is a backend application that helps users not only check the weather, but also plan trips based on weather conditions.

### Main features
- Check current weather by location
- Plan a trip for a specific date
- Plan a trip for a date range
- Store user requests in a database
- Perform CRUD operations
- Export data in CSV and JSON formats
- Generate Google Maps links
- Provide travel risk analysis and packing advice

This project transforms a simple weather app into a travel decision assistant.

In [ ]:
!pip install requests pandas

## 📦 Import Libraries

In this section, we import the libraries needed for:
- API calls
- database management
- data export
- date validation

In [ ]:
import requests
import sqlite3
import pandas as pd
import json
from datetime import datetime

## 🗄️ Database Setup

We create a local SQLite database to store all user weather and travel requests.

The database will contain:
- location information
- travel dates
- weather data
- travel risk
- packing advice
- map links

In [ ]:
conn = sqlite3.connect("weather_travel.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS weather_requests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    location_query TEXT,
    city TEXT,
    country TEXT,
    latitude REAL,
    longitude REAL,
    request_type TEXT,
    single_date TEXT,
    start_date TEXT,
    end_date TEXT,
    avg_temp_c REAL,
    max_temp_c REAL,
    min_temp_c REAL,
    weather_summary TEXT,
    travel_risk TEXT,
    advice TEXT,
    maps_url TEXT,
    created_at TEXT
)
""")

conn.commit()

print("Database and table ready.")

Database and table ready.


## ⚙️ Utility Functions

These helper functions are used to:
- validate dates
- convert weather codes into readable text
- assign weather icons
- calculate travel risk
- generate practical travel advice

In [ ]:
def validate_date(date_text):
    try:
        datetime.strptime(date_text, "%Y-%m-%d")
        return True
    except ValueError:
        return False


def weather_icon(summary):
    summary = summary.lower()

    if "clear" in summary or "sun" in summary:
        return "☀️"
    elif "rain" in summary or "drizzle" in summary:
        return "🌧️"
    elif "cloud" in summary or "overcast" in summary:
        return "☁️"
    elif "snow" in summary:
        return "❄️"
    elif "fog" in summary or "mist" in summary:
        return "🌫️"
    elif "storm" in summary or "thunder" in summary:
        return "⛈️"
    else:
        return "🌍"


def weather_code_to_text(code):
    weather_codes = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        61: "Light rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Light snow",
        73: "Moderate snow",
        75: "Heavy snow",
        80: "Rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        95: "Thunderstorm"
    }
    return weather_codes.get(code, "Unknown weather")


def calculate_travel_risk(avg_temp, summary):
    summary = summary.lower()

    if "thunder" in summary or "violent" in summary or "heavy rain" in summary:
        return "High"
    elif "rain" in summary or "snow" in summary or avg_temp < 5 or avg_temp > 35:
        return "Medium"
    else:
        return "Low"


def generate_advice(avg_temp, summary):
    summary = summary.lower()

    if "rain" in summary:
        return "Bring an umbrella ☂️"
    elif "snow" in summary:
        return "Wear a warm coat and winter shoes 🧥"
    elif avg_temp > 30:
        return "Wear light clothes and stay hydrated 🧴"
    elif avg_temp < 10:
        return "Bring warm layers 🧣"
    else:
        return "Weather looks good for travel 👍"

## 📍 Geocoding: Convert Location Name into Coordinates

Users enter a location name such as a city or town.

The geocoding API converts this input into:
- city
- country
- latitude
- longitude

These coordinates are required to retrieve weather data accurately.

In [ ]:
def geocode_location(location_query):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": location_query,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "results" in data and len(data["results"]) > 0:
        result = data["results"][0]
        return {
            "city": result.get("name"),
            "country": result.get("country"),
            "latitude": result.get("latitude"),
            "longitude": result.get("longitude")
        }
    else:
        return None

## 🌤️ Current Weather Retrieval

This function retrieves the current weather for a selected location using its coordinates.

*   Elemento de lista
*   Elemento de lista



In [ ]:
def get_current_weather(latitude, longitude):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,weather_code",
        "timezone": "auto"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "current" not in data:
        return None

    current = data["current"]
    temp = current.get("temperature_2m")
    code = current.get("weather_code")
    summary = weather_code_to_text(code)

    return {
        "avg_temp_c": temp,
        "max_temp_c": temp,
        "min_temp_c": temp,
        "weather_summary": summary
    }

## 📅 Weather for a Single Date or Date Range

This function retrieves daily weather information for:
- one specific date
- a range of travel dates

It calculates:
- average temperature
- maximum temperature
- minimum temperature
- weather summary

In [ ]:
def get_weather_for_dates(latitude, longitude, start_date, end_date):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": "temperature_2m_max,temperature_2m_min,weather_code",
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "daily" not in data:
        return None

    daily = data["daily"]

    max_temps = daily.get("temperature_2m_max", [])
    min_temps = daily.get("temperature_2m_min", [])
    codes = daily.get("weather_code", [])

    if not max_temps or not min_temps or not codes:
        return None

    avg_temp = round(
        sum([(mx + mn) / 2 for mx, mn in zip(max_temps, min_temps)]) / len(max_temps),
        2
    )
    max_temp = max(max_temps)
    min_temp = min(min_temps)

    most_common_code = max(set(codes), key=codes.count)
    summary = weather_code_to_text(most_common_code)

    return {
        "avg_temp_c": avg_temp,
        "max_temp_c": max_temp,
        "min_temp_c": min_temp,
        "weather_summary": summary
    }

## 💾 Save Request to the Database

This function stores each request in the database.

It saves:
- location
- weather data
- request type
- travel dates
- travel risk
- advice
- map link

In [ ]:
def save_request(record):
    cursor.execute("""
    INSERT INTO weather_requests (
        location_query, city, country, latitude, longitude,
        request_type, single_date, start_date, end_date,
        avg_temp_c, max_temp_c, min_temp_c,
        weather_summary, travel_risk, advice, maps_url, created_at
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        record["location_query"],
        record["city"],
        record["country"],
        record["latitude"],
        record["longitude"],
        record["request_type"],
        record["single_date"],
        record["start_date"],
        record["end_date"],
        record["avg_temp_c"],
        record["max_temp_c"],
        record["min_temp_c"],
        record["weather_summary"],
        record["travel_risk"],
        record["advice"],
        record["maps_url"],
        record["created_at"]
    ))
    conn.commit()

## 📖 Read Saved Requests

This function displays all saved requests from the database.

It supports the READ operation from CRUD.

In [ ]:
def show_all_requests():
    df = pd.read_sql_query("SELECT * FROM weather_requests", conn)

    if df.empty:
        print("No saved requests found.")
    else:
        print(df)

## ✏️ Update Existing Requests

This function allows users to update:
- a single travel date
- a date range

It supports the UPDATE operation from CRUD.

In [ ]:
def update_request():
    record_id = input("Enter record ID to update: ").strip()

    new_single_date = input("Enter new single date (YYYY-MM-DD) or leave blank: ").strip()
    new_start_date = input("Enter new start date (YYYY-MM-DD) or leave blank: ").strip()
    new_end_date = input("Enter new end date (YYYY-MM-DD) or leave blank: ").strip()

    if new_single_date and not validate_date(new_single_date):
        print("Invalid single date format.")
        return

    if new_start_date and not validate_date(new_start_date):
        print("Invalid start date format.")
        return

    if new_end_date and not validate_date(new_end_date):
        print("Invalid end date format.")
        return

    if new_start_date and new_end_date and new_start_date > new_end_date:
        print("Invalid date range: start date must be before or equal to end date.")
        return

    cursor.execute("SELECT * FROM weather_requests WHERE id = ?", (record_id,))
    row = cursor.fetchone()

    if row is None:
        print("Record not found.")
        return

    if new_single_date:
        cursor.execute("""
        UPDATE weather_requests
        SET single_date = ?, start_date = NULL, end_date = NULL, request_type = 'single_date'
        WHERE id = ?
        """, (new_single_date, record_id))

    if new_start_date or new_end_date:
        if new_start_date:
            cursor.execute("""
            UPDATE weather_requests
            SET start_date = ?, single_date = NULL, request_type = 'date_range'
            WHERE id = ?
            """, (new_start_date, record_id))

        if new_end_date:
            cursor.execute("""
            UPDATE weather_requests
            SET end_date = ?, single_date = NULL, request_type = 'date_range'
            WHERE id = ?
            """, (new_end_date, record_id))

    conn.commit()
    print("Record updated successfully.")

## 🗑️ Delete Requests

This function allows users to remove a saved request from the database.

It supports the DELETE operation from CRUD.

In [ ]:
def delete_request():
    record_id = input("Enter record ID to delete: ").strip()

    cursor.execute("SELECT * FROM weather_requests WHERE id = ?", (record_id,))
    row = cursor.fetchone()

    if row is None:
        print("Record not found.")
        return

    confirm = input("Are you sure you want to delete this record? (yes/no): ").strip().lower()

    if confirm == "yes":
        cursor.execute("DELETE FROM weather_requests WHERE id = ?", (record_id,))
        conn.commit()
        print("Record deleted successfully.")
    else:
        print("Delete operation cancelled.")

## 🚀 Main Interactive Menu

This is the main application flow.

Users can:
1. Check current weather
2. Plan a trip for a single date
3. Plan a trip for a date range
4. View saved requests
5. Update a saved request
6. Delete a saved request
7. Exit the application

In [ ]:
def main_menu():
    while True:
        print("\n=== Weather Travel Assistant ===")
        print("1. Check current weather")
        print("2. Plan trip for a single date")
        print("3. Plan trip for a date range")
        print("4. View saved requests")
        print("5. Update a saved request")
        print("6. Delete a saved request")
        print("7. Exit")

        choice = input("Choose an option (1-7): ").strip()

        if choice == "1":
            location_query = input("Enter a location: ").strip()
            location_data = geocode_location(location_query)

            if not location_data:
                print("Location not found.")
                continue

            city = location_data["city"]
            country = location_data["country"]
            latitude = location_data["latitude"]
            longitude = location_data["longitude"]

            weather_data = get_current_weather(latitude, longitude)

            if not weather_data:
                print("Could not retrieve current weather data.")
                continue

            summary = weather_data["weather_summary"]
            avg_temp = weather_data["avg_temp_c"]
            risk = calculate_travel_risk(avg_temp, summary)
            advice = generate_advice(avg_temp, summary)
            icon = weather_icon(summary)
            maps_url = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"

            record = {
                "location_query": location_query,
                "city": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude,
                "request_type": "current",
                "single_date": None,
                "start_date": None,
                "end_date": None,
                "avg_temp_c": avg_temp,
                "max_temp_c": weather_data["max_temp_c"],
                "min_temp_c": weather_data["min_temp_c"],
                "weather_summary": summary,
                "travel_risk": risk,
                "advice": advice,
                "maps_url": maps_url,
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            print("\n--- CURRENT WEATHER ---")
            print(f"Location: {city}, {country}")
            print(f"Weather: {icon} {summary}")
            print(f"Temperature: {avg_temp}°C")
            print(f"Travel risk: {risk}")
            print(f"Advice: {advice}")
            print(f"Map: {maps_url}")

            save_request(record)
            print("Request saved successfully.")

        elif choice == "2":
            location_query = input("Enter a location: ").strip()
            single_date = input("Enter travel date (YYYY-MM-DD): ").strip()

            if not validate_date(single_date):
                print("Invalid date format. Use YYYY-MM-DD.")
                continue

            location_data = geocode_location(location_query)

            if not location_data:
                print("Location not found.")
                continue

            city = location_data["city"]
            country = location_data["country"]
            latitude = location_data["latitude"]
            longitude = location_data["longitude"]

            weather_data = get_weather_for_dates(latitude, longitude, single_date, single_date)

            if not weather_data:
                print("Could not retrieve weather data for that date.")
                continue

            summary = weather_data["weather_summary"]
            avg_temp = weather_data["avg_temp_c"]
            risk = calculate_travel_risk(avg_temp, summary)
            advice = generate_advice(avg_temp, summary)
            icon = weather_icon(summary)
            maps_url = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"

            record = {
                "location_query": location_query,
                "city": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude,
                "request_type": "single_date",
                "single_date": single_date,
                "start_date": None,
                "end_date": None,
                "avg_temp_c": avg_temp,
                "max_temp_c": weather_data["max_temp_c"],
                "min_temp_c": weather_data["min_temp_c"],
                "weather_summary": summary,
                "travel_risk": risk,
                "advice": advice,
                "maps_url": maps_url,
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            print("\n--- SINGLE DATE TRAVEL PLAN ---")
            print(f"Location: {city}, {country}")
            print(f"Date: {single_date}")
            print(f"Weather: {icon} {summary}")
            print(f"Average temperature: {avg_temp}°C")
            print(f"Max temperature: {weather_data['max_temp_c']}°C")
            print(f"Min temperature: {weather_data['min_temp_c']}°C")
            print(f"Travel risk: {risk}")
            print(f"Advice: {advice}")
            print(f"Map: {maps_url}")

            save_request(record)
            print("Request saved successfully.")

        elif choice == "3":
            location_query = input("Enter a location: ").strip()
            start_date = input("Enter start date (YYYY-MM-DD): ").strip()
            end_date = input("Enter end date (YYYY-MM-DD): ").strip()

            if not validate_date(start_date) or not validate_date(end_date):
                print("Invalid date format. Use YYYY-MM-DD.")
                continue

            if start_date > end_date:
                print("Invalid date range: start date must be before or equal to end date.")
                continue

            location_data = geocode_location(location_query)

            if not location_data:
                print("Location not found.")
                continue

            city = location_data["city"]
            country = location_data["country"]
            latitude = location_data["latitude"]
            longitude = location_data["longitude"]

            weather_data = get_weather_for_dates(latitude, longitude, start_date, end_date)

            if not weather_data:
                print("Could not retrieve weather data for that date range.")
                continue

            summary = weather_data["weather_summary"]
            avg_temp = weather_data["avg_temp_c"]
            risk = calculate_travel_risk(avg_temp, summary)
            advice = generate_advice(avg_temp, summary)
            icon = weather_icon(summary)
            maps_url = f"https://www.google.com/maps/search/?api=1&query={latitude},{longitude}"

            record = {
                "location_query": location_query,
                "city": city,
                "country": country,
                "latitude": latitude,
                "longitude": longitude,
                "request_type": "date_range",
                "single_date": None,
                "start_date": start_date,
                "end_date": end_date,
                "avg_temp_c": avg_temp,
                "max_temp_c": weather_data["max_temp_c"],
                "min_temp_c": weather_data["min_temp_c"],
                "weather_summary": summary,
                "travel_risk": risk,
                "advice": advice,
                "maps_url": maps_url,
                "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }

            print("\n--- DATE RANGE TRAVEL PLAN ---")
            print(f"Location: {city}, {country}")
            print(f"Dates: {start_date} to {end_date}")
            print(f"Weather: {icon} {summary}")
            print(f"Average temperature: {avg_temp}°C")
            print(f"Max temperature: {weather_data['max_temp_c']}°C")
            print(f"Min temperature: {weather_data['min_temp_c']}°C")
            print(f"Travel risk: {risk}")
            print(f"Advice: {advice}")
            print(f"Map: {maps_url}")

            save_request(record)
            print("Request saved successfully.")

        elif choice == "4":
            show_all_requests()

        elif choice == "5":
            show_all_requests()
            update_request()

        elif choice == "6":
            show_all_requests()
            delete_request()

        elif choice == "7":
            print("Exiting application.")
            break

        else:
            print("Invalid option. Please choose a number from 1 to 7.")

## ▶️ Run the Application

The following command starts the interactive backend menu.

In [ ]:
main_menu()


=== Weather Travel Assistant ===
1. Check current weather
2. Plan trip for a single date
3. Plan trip for a date range
4. View saved requests
5. Update a saved request
6. Delete a saved request
7. Exit

--- DATE RANGE TRAVEL PLAN ---
Location: Paris, France
Dates: 2026-03-23 to 2026-03-30
Weather: 🌧️ Light rain
Average temperature: 9.15°C
Max temperature: 18.6°C
Min temperature: 1.7°C
Travel risk: Medium
Advice: Bring an umbrella ☂️
Map: https://www.google.com/maps/search/?api=1&query=48.85341,2.3488
Request saved successfully.

=== Weather Travel Assistant ===
1. Check current weather
2. Plan trip for a single date
3. Plan trip for a date range
4. View saved requests
5. Update a saved request
6. Delete a saved request
7. Exit


## 📤 Data Export

The application can export all saved requests into:
- CSV
- JSON

This supports portability and reporting.

In [ ]:
def export_data():
    df = pd.read_sql_query("SELECT * FROM weather_requests", conn)

    df.to_csv("weather_requests.csv", index=False)

    with open("weather_requests.json", "w") as f:
        json.dump(df.to_dict(orient="records"), f, indent=4)

    print("Data exported successfully to CSV and JSON.")

In [ ]:
export_data()

In [ ]:
from google.colab import files

files.download("weather_requests.csv")
files.download("weather_requests.json")